In [11]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint,HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.vectorstores import FAISS

In [10]:
from dotenv import load_dotenv

1. Indexing

Documnet Ingestion

In [5]:
video_id="Gfr50f6ZBvo"  # only the video id not the complete url
yt_api=YouTubeTranscriptApi()
try:
    # if u dont specify lang then it returns best one
    transcript_list=yt_api.fetch(video_id,languages=['en'])

    # Flatten it into plain text
    transcript=" ".join(chunk.text for chunk in transcript_list)

except TranscriptsDisabled:
    print('NO captions are available for this video')


Text Splitting

In [6]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [8]:
len(chunks)

168

In [9]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

Embeddings and Vector Store

In [12]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.from_documents(chunks,embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4061.92it/s]


2. Retriever

In [14]:
retriever=vector_store.as_retriever(search_type='similarity',search_kwargs={'k':4})

In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000014888E50100>, search_kwargs={'k': 4})

In [16]:
retriever.invoke('What is Deep Mind')

[Document(id='63c7d987-7e1d-4b7a-a3d5-00a19eb37d0f', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

3. Augmentation

In [18]:
llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm,temperature=0.2)

In [19]:
prompt=PromptTemplate(
    template="""You are a helpful assistant.
    Answer ONLY from th provided transcript context.
    If the context is insufficient , just say you don't know.
    
    {context}
    Question: {question}""",
    input_variables=['context','question']
)

In [ ]:
question="is the topic of aliens discussed in this video? if yes what is discussed"
retrieved_docs=retriever.invoke(question)
context_text='\n\n'.join(doc.page_content for doc in retrieved_docs)

In [23]:
retrieved_docs

[Document(id='97dcef24-0c84-4327-9cab-cf84e0441c58', metadata={}, page_content="thoughts it could be some interactions with our mind that we think are originating from us is actually something that uh is coming from other life forms elsewhere consciousness itself might be that it could be but i don't see any sensible argument to the why why would all of the alien species be using this way yes some of them will be more primitive they would be close to our level you know there would there should be a whole sort of normal distribution of these things right some would be aggressive some would be you know curious others would be very stoical and philosophical because you know maybe they're a million years older than us but it's not it shouldn't be like what i mean one one alien civilization might be like that communicating thoughts and others but i don't see why you know potentially the hundreds there should be would be uniform in this way right it could be a violent dictatorship that the t

In [21]:
final_prompt=prompt.invoke({'context':context_text,'question':question})

In [24]:
final_prompt

StringPromptValue(text="You are a helpful assistant.\n    Answer ONLY from th provided transcript context.\n    If the context is insufficient , just say you don't know.\n    \n    thoughts it could be some interactions with our mind that we think are originating from us is actually something that uh is coming from other life forms elsewhere consciousness itself might be that it could be but i don't see any sensible argument to the why why would all of the alien species be using this way yes some of them will be more primitive they would be close to our level you know there would there should be a whole sort of normal distribution of these things right some would be aggressive some would be you know curious others would be very stoical and philosophical because you know maybe they're a million years older than us but it's not it shouldn't be like what i mean one one alien civilization might be like that communicating thoughts and others but i don't see why you know potentially the hund

4. Generation

In [25]:
answer=model.invoke(final_prompt)
print(answer)

content='Yes, the topic of aliens is discussed in this video. The discussion revolves around the possibility of alien civilizations, their potential communication methods (such as thought transmission), and the lack of evidence for their existence. The speaker also mentions the Fermi Paradox and the possibility of a "great filter" that prevents civilizations from becoming multi-planetary or reaching out into the stars.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 878, 'total_tokens': 953}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': 'vllm-0.21.0-dbb32ef8', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f5ac1-625d-73c3-8ef9-e5a012542f52-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 878, 'output_tokens': 75, 'total_tokens': 953}


Buiding the Complete Chain

In [26]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [27]:
def format_docs(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [28]:
parallel_chain=RunnableParallel({
    'context':retriever|RunnableLambda(format_docs),
    'question':RunnablePassthrough()
    })

In [31]:
parallel_chain.invoke('Who is Demis Hasabis')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [32]:
parser=StrOutputParser()

In [33]:
main_chain=parallel_chain|prompt|model|parser 

In [34]:
main_chain.invoke('Can you summarize the video')

"The conversation is about explaining complex topics in simple terms, particularly in the context of physics and AI. The speaker mentions that a deeper, simpler explanation of the world is needed, one that encompasses mysteries such as consciousness, life, and gravity. They also discuss the importance of being able to explain complex topics clearly and simply, and how this is a sign of intelligence. Additionally, the conversation touches on the history of AI, including Claude Shannon's first chess program and the development of Deep Blue, which beat a human chess champion. The speaker reflects on how they were more impressed by the human mind of the champion, Garry Kasparov, than by the machine's abilities."